In [ ]:
print ("jupyter is working")


In [ ]:
pip install psycopg2-binary

In [ ]:
pip install requests pandas

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()
API_KEY = os.getenv("API_KEY")
API_HOST = "aerodatabox.p.rapidapi.com"
HEADERS = {
    "x-rapidapi-key": API_KEY,
    "x-rapidapi-host": API_HOST
}
DB_CONFIG = {
    "host": "localhost",
    "port": "5432",
    "database": "air_tracker",
    "user": "postgres",
    "password": os.getenv("DB_PASSWORD")
}
print("Config ready!")

In [ ]:
import requests
import pandas as pd
import psycopg2
import json
from datetime import datetime, timedelta
print("All libraries imported successfully!")

In [ ]:
def get_connection():
    conn = psycopg2.connect(
        host=DB_CONFIG["host"],
        port=DB_CONFIG["port"],
        database=DB_CONFIG["database"],
        user=DB_CONFIG["user"],
        password=DB_CONFIG["password"]
    )
    return conn

# Test the connection
try:
    conn = get_connection()
    print("Connected to PostgreSQL successfully!")
    conn.close()
except Exception as e:
    print(f"Connection failed: {e}")

In [ ]:
import time
AIRPORT_CODES = [
    "DXB", "SIN", "LHR", "JFK", "SYD", "DOH", "BKK", "CDG",
    "MAA", "DEL", "BOM", "BLR", "HYD", "CCU", "COK"
]
def fetch_airport(iata_code):
    url = f"https://{API_HOST}/airports/iata/{iata_code}"
    try:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Error fetching {iata_code}: {e}")
        return None
airports_data = []
for code in AIRPORT_CODES:
    print(f"Fetching {code}...")
    data = None
    attempts = 0
    while data is None and attempts < 3:
        data = fetch_airport(code)
        if data is None:
            time.sleep(3)
        attempts += 1
    if data:
        airports_data.append(data)
    time.sleep(2)
print(f"\nSuccessfully fetched {len(airports_data)} airports!")

In [ ]:
print(json.dumps(airports_data[0], indent=2))

In [ ]:
# Insert airport data into PostgreSQL
def insert_airports(airports):
    conn = get_connection()
    cursor = conn.cursor()
    
    for airport in airports:
        try:
            cursor.execute("""
                INSERT INTO airport 
                (icao_code, iata_code, name, city, country, continent, latitude, longitude, timezone)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
                ON CONFLICT (iata_code) DO NOTHING
            """, (
                airport.get("icao"),
                airport.get("iata"),
                airport.get("fullName"),
                airport.get("municipalityName"),
                airport.get("country", {}).get("name"),
                airport.get("continent", {}).get("name"),
                airport.get("location", {}).get("lat"),
                airport.get("location", {}).get("lon"),
                airport.get("timeZone")
            ))
        except Exception as e:
            print(f"Error inserting {airport.get('iata')}: {e}")
    
    conn.commit()
    cursor.close()
    conn.close()
    print(f"Inserted {len(airports)} airports successfully!")

insert_airports(airports_data)

In [ ]:
def fetch_flights(iata_code, date):
    url = f"https://{API_HOST}/flights/airports/iata/{iata_code}"
    params = {
        "fromLocal": f"{date}T00:00",
        "toLocal": f"{date}T23:59",
        "withLeg": "true",
        "direction": "Both",
        "withCancelled": "true",
        "withCodeshared": "false",
        "withCargo": "false",
        "withPrivate": "false"
    }
    try:
        response = requests.get(url, headers=HEADERS, params=params)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Error fetching flights for {iata_code}: {e}")
        return None

# Test with MAA for yesterday
yesterday = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
print(f"Fetching flights for MAA on {yesterday}...")
test_flights = fetch_flights("MAA", yesterday)
print(json.dumps(test_flights, indent=2))

In [ ]:
# Fetch and insert flights for all airports
def insert_flights(flights_list, airport_iata):
    conn = get_connection()
    cursor = conn.cursor()
    inserted = 0
    
    for flight in flights_list:
        try:
            cursor.execute("""
                INSERT INTO flights 
                (flight_id, flight_number, aircraft_model, origin_iata, destination_iata,
                scheduled_departure, actual_departure, scheduled_arrival, actual_arrival,
                status, airline_code)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                ON CONFLICT (flight_id) DO NOTHING
            """, (
                flight.get("number"),
                flight.get("number"),
                flight.get("aircraft", {}).get("model"),
                flight.get("departure", {}).get("airport", {}).get("iata", airport_iata),
                flight.get("arrival", {}).get("airport", {}).get("iata"),
                flight.get("departure", {}).get("scheduledTime", {}).get("local"),
                flight.get("departure", {}).get("actualTime", {}).get("local"),
                flight.get("arrival", {}).get("scheduledTime", {}).get("local"),
                flight.get("arrival", {}).get("actualTime", {}).get("local"),
                flight.get("status"),
                flight.get("airline", {}).get("iata")
            ))
            inserted += 1
        except Exception as e:
            print(f"Error inserting flight {flight.get('number')}: {e}")
    
    conn.commit()
    cursor.close()
    conn.close()
    return inserted

# Fetch flights for all 15 airports
yesterday = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
total_flights = 0

for code in AIRPORT_CODES:
    print(f"Fetching flights for {code}...")
    data = fetch_flights(code, yesterday)
    if data:
        flights_list = []
        if "departures" in data:
            flights_list += data["departures"]
        if "arrivals" in data:
            flights_list += data["arrivals"]
        count = insert_flights(flights_list, code)
        total_flights += count
        print(f"  → Inserted {count} flights")
    time.sleep(2)

print(f"\nTotal flights inserted: {total_flights}")

In [ ]:
# Test with different airline name formats
for name in ["KLM", "indigo", "6E", "IGO", "emirates", "Emirates"]:
    result = fetch_airline_aircraft(name)
    if result:
        print(f"✓ Works with: {name}")
        print(json.dumps(result, indent=2))
        break
    else:
        print(f"✗ Failed with: {name}")
    time.sleep(1)

In [ ]:
# Use IATA codes directly for aircraft fetch
TOP_AIRLINES = ["6E", "AF", "AI", "BA", "QF", "EK", "IX", "SQ", "QR", "DL"]

total_aircraft = 0
for code in TOP_AIRLINES:
    print(f"Fetching fleet for {code}...")
    data = fetch_airline_aircraft(code)
    if data and "items" in data:
        count = insert_aircraft(data["items"], code)
        total_aircraft += count
        print(f"  → Inserted {count} aircraft")
    else:
        print(f"  → No data found")
    time.sleep(2)

print(f"\nTotal aircraft inserted: {total_aircraft}")

In [ ]:
# Test airport delays with ICAO code
def fetch_airport_delays(icao_code):
    url = f"https://{API_HOST}/airports/icao/{icao_code}/delays"
    try:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Error fetching delays for {icao_code}: {e}")
        return None

# Test with MAA (ICAO: VOMM)
test_delays = fetch_airport_delays("VOMM")
print(json.dumps(test_delays, indent=2))

In [ ]:
# Airport ICAO to IATA mapping
AIRPORT_ICAO = {
    "DXB": "OMDB", "SIN": "WSSS", "LHR": "EGLL", "JFK": "KJFK",
    "SYD": "YSSY", "DOH": "OTHH", "BKK": "VTBS", "CDG": "LFPG",
    "MAA": "VOMM", "DEL": "VIDP", "BOM": "VABB", "BLR": "VOBL",
    "HYD": "VOHS", "CCU": "VECC", "COK": "VOCI"
}

def insert_delays(iata_code, data):
    conn = get_connection()
    cursor = conn.cursor()
    try:
        total = (data.get("departuresDelayInformation", {}).get("numTotal", 0) +
                 data.get("arrivalsDelayInformation", {}).get("numTotal", 0))
        delayed = (data.get("departuresDelayInformation", {}).get("numQualifiedTotal", 0) +
                   data.get("arrivalsDelayInformation", {}).get("numQualifiedTotal", 0))
        cancelled = (data.get("departuresDelayInformation", {}).get("numCancelled", 0) +
                     data.get("arrivalsDelayInformation", {}).get("numCancelled", 0))
        cursor.execute("""
            INSERT INTO airport_delays 
            (airport_iata, delay_date, total_flights, delayed_flights, avg_delay_min, median_delay_min, canceled_flights)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
        """, (
            iata_code,
            data.get("from", {}).get("utc"),
            total,
            delayed,
            0,
            0,
            cancelled
        ))
        conn.commit()
        print(f"  → Inserted delays for {iata_code}")
    except Exception as e:
        print(f"Error inserting delays for {iata_code}: {e}")
    cursor.close()
    conn.close()

# Fetch delays for all 15 airports
for iata, icao in AIRPORT_ICAO.items():
    print(f"Fetching delays for {iata}...")
    data = fetch_airport_delays(icao)
    if data:
        insert_delays(iata, data)
    time.sleep(2)

print("\nAll delays inserted!")